In [2]:
import pandas as pd
import re
import matplotlib.pyplot as plt
import os
from ast import literal_eval
from collections import Counter
import math
import numpy as np
import seaborn as sns

In [3]:
username = os.path.expanduser('~')
username

'/home/ehdgus7279'

In [4]:
benchmarks = [
    # High-sharing Apps
    'bfs',
    'gemm',
    'lu',
    'spmv',
    'mm3',
    'mm2',
    'matrixmultiplication',
    'syr2k',
    'gups',    
    'mis',
    'matrixtranspose',
    'stencil2d',
    'gaussian',
    'im2col',
    'vgg16',

    # Low-sharing Apps
    'fdtd2d',
    'fastwalshtransform',
    'jacobi1d',
    'relu',
    'color',
    'convolution2d',
    'sssp',
    'fir',
    'fft',
    'bitonicsort',
    'pagerank',
]

In [5]:
num_chiplets = 36
ns = 0.000000001
chip_cols = [str(i) for i in range(36)]

In [6]:
base_dir=f'{username}/mgvm/results/memory-traces'
date = "20260329-231025-mesh"
expanded_dir = f"{base_dir}/{date}/expanded"
trace_dir = f"{base_dir}/{date}"
print(f"{base_dir}/{date}")

/home/ehdgus7279/mgvm/results/memory-traces/20260329-231025-mesh


In [13]:
benchmark = "gemm"

trace_df = pd.read_csv(f"{expanded_dir}/{benchmark}_mesh_6x6_memory_trace.csv")
trace_df

,time,pc,addr,owner,rw,0,1,2,3,4,...,28,29,30,31,32,33,34,35,access_count,shared_chiplets
0,0.012190,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
1,0.012190,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
2,0.012190,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
3,0.012190,300011776,2447167488,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
4,0.012190,300011776,2447167488,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24776432,0.013472,300012060,2282071360,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
24776433,0.013472,300012060,2282071424,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1
24776434,0.013472,300012060,2263185024,1,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,1,1
24776435,0.013472,300012060,2324491008,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,1,1


In [14]:
def get_rdma_df(df):
    rdma_df = df.copy()
    for i in range(num_chiplets):
        col = str(i)
        rdma_df[col] = np.where(rdma_df['owner'] == i, 0, rdma_df[col])
    rdma_df['is_rdma'] = rdma_df[chip_cols].any(axis=1)
    return rdma_df[rdma_df['is_rdma']]

In [15]:
rdma_df = get_rdma_df(trace_df)

In [16]:
dup_pcs = rdma_df.groupby('addr').agg({
    'pc': 'nunique',
})

dup_pcs.mean()

pc    1.711803
dtype: float64